# 🤖 Chapter 6: Statistical Machine Learning
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 6.1 Introduction: Statistics vs. Machine Learning
In classical statistics (like Linear or Logistic Regression), we start with a predefined mathematical equation and try to find the weights that best fit the data. The goal is often *inference*—understanding exactly how each variable impacts the outcome.

**Statistical Machine Learning** takes a different approach. Instead of assuming a rigid mathematical equation, these algorithms are data-driven. They organically learn complex, non-linear boundaries and interactions directly from the data. The primary goal is *prediction accuracy*.

In this chapter, we will cover the four pillars of statistical machine learning:
1. **K-Nearest Neighbors (KNN):** Predicting based on similarity to past examples.
2. **Tree Models:** Splitting data using a flowchart of rules.
3. **Bagging and Random Forests:** Averaging hundreds of trees to prevent overfitting.
4. **Boosting:** Training trees sequentially to correct the mistakes of previous trees.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 6.")

## 6.2 Creating a Synthetic Customer Dataset
To demonstrate these algorithms, we will create a dataset representing e-commerce customers. 
Our goal is to predict the `Purchase` variable (1 = Bought, 0 = Did not buy) based on three features: `Age`, `Income`, and `Time_on_Site`.

We will intentionally inject non-linear relationships (e.g., both very young and very old people might buy less, middle-aged people buy more) which linear regression would struggle to capture.

In [ ]:
# Generate Synthetic Customer Data
n_samples = 1500

age = np.random.uniform(18, 70, n_samples)
income = np.random.normal(60000, 20000, n_samples)
time_on_site = np.random.exponential(5, n_samples)  # Minutes

# Create a complex, non-linear rule for purchasing
# High likelihood to buy if: (Age between 30 and 50) AND (Income > 50k) AND (Time > 3 mins)
prob_purchase = np.zeros(n_samples)

for i in range(n_samples):
    base_prob = 0.1
    if 30 < age[i] < 50:
        base_prob += 0.3
    if income[i] > 50000:
        base_prob += 0.2
    if time_on_site[i] > 3:
        base_prob += 0.3
    
    # Cap probability at 0.95
    prob_purchase[i] = min(base_prob, 0.95)

# Generate actual target classes based on probabilities
purchase = np.random.binomial(1, prob_purchase)

df = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Time_on_Site': time_on_site,
    'Purchase': purchase
})

# Split into Train and Test
X = df[['Age', 'Income', 'Time_on_Site']]
y = df['Purchase']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset generated. Baseline Purchase Rate: {np.mean(purchase)*100:.1f}%")

## 6.3 K-Nearest Neighbors (KNN)
KNN is conceptually the simplest machine learning algorithm. To predict a new record, it looks at the $K$ most similar records in the training data and takes a majority vote.

**The Standardization Trap:**
KNN relies on measuring physical distance (usually Euclidean distance). If `Income` ranges from 20,000 to 100,000, and `Time_on_Site` ranges from 1 to 15, the `Income` variable will completely dominate the distance calculation simply because its numbers are larger. 

To fix this, we MUST standardize (Z-score scaling) the data before using KNN, so all features have a mean of 0 and a standard deviation of 1.

In [ ]:
# 1. Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train KNN Model (Let's use K=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# 3. Evaluate
knn_preds = knn.predict(X_test_scaled)
print("--- K-Nearest Neighbors (K=5) ---")
print(f"Accuracy: {accuracy_score(y_test, knn_preds):.3f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, knn_preds))

## 6.4 Tree Models (Decision Trees)
Tree models (also known as Classification and Regression Trees - CART) attempt to split the data using a series of if-then rules. 

The algorithm searches through every variable and every possible split point to find the rule that divides the data into the most "pure" child nodes (e.g., separating Buyers from Non-Buyers as cleanly as possible). The mathematical metric used to measure this purity is usually **Gini Impurity** or **Entropy**.

**Advantages:** Trees do not require scaling, they handle non-linear data perfectly, and they are highly interpretable.
**Disadvantages:** A single tree is highly prone to *overfitting* (memorizing the training data noise).

In [ ]:
# Train a Decision Tree Classifier
# We limit max_depth to 3 to prevent overfitting and keep the plot readable
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

tree_preds = tree.predict(X_test)
print("--- Decision Tree (Max Depth = 3) ---")
print(f"Accuracy: {accuracy_score(y_test, tree_preds):.3f}")

# Visualizing the Tree
plt.figure(figsize=(16, 8))
plot_tree(tree, feature_names=X.columns, class_names=['No Buy', 'Buy'], filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Flowchart")
plt.show()

print("Notice how the tree organically discovers the rules we coded into the synthetic data (e.g., splitting on Time_on_Site and Age)!")

## 6.5 Bagging and the Random Forest
To solve the overfitting problem of a single decision tree, we use an ensemble method called **Random Forest**.

It relies on two powerful statistical concepts:
1. **Bagging (Bootstrap Aggregating):** It builds hundreds of distinct decision trees. Each tree is trained on a different random *Bootstrap Sample* of the original data.
2. **Random Feature Selection:** At every split in every tree, the algorithm is only allowed to look at a random subset of features. This prevents a single dominant feature from controlling every single tree.

By having hundreds of "weak" trees cast a vote, the Random Forest becomes incredibly robust, highly accurate, and resistant to overfitting.

In [ ]:
# Train a Random Forest (Ensemble of 100 trees)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
print("--- Random Forest (100 Trees) ---")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.3f}")

# Variable Importance
# Unlike a single tree, we can't draw a flowchart for a Random Forest.
# Instead, we look at 'Feature Importance' to see which variable contributed the most to decreasing Gini Impurity.
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=importance.values, y=importance.index, palette='viridis')
plt.title('Random Forest Feature Importance')
plt.xlabel('Mean Decrease in Impurity')
plt.show()

## 6.6 Boosting (Gradient Boosting)
While Random Forest builds hundreds of trees independently and averages them, **Boosting** builds trees *sequentially*.

In Gradient Boosting:
1. Tree 1 tries to predict the target.
2. Tree 2 is trained specifically to predict the *errors (residuals)* made by Tree 1.
3. Tree 3 is trained to predict the errors made by Tree 2, and so on.

To prevent the model from learning too fast and overfitting, Boosting uses a **Learning Rate** (shrinkage parameter). It only adds a small fraction of each tree's prediction to the final tally.

Boosting algorithms (like XGBoost, LightGBM, and standard Gradient Boosting) are widely considered the most powerful algorithms for structured, tabular data.

In [ ]:
# Train a Gradient Boosting Machine (GBM)
# learning_rate shrinks the contribution of each tree, requiring more trees (n_estimators) to compensate
gbm = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

gbm_preds = gbm.predict(X_test)
print("--- Gradient Boosting Machine ---")
print(f"Accuracy: {accuracy_score(y_test, gbm_preds):.3f}")

# Detailed Classification Report for the best model
print("\nDetailed GBM Classification Report:")
print(classification_report(y_test, gbm_preds, target_names=['No Buy (0)', 'Buy (1)']))

### Conclusion of Chapter 6
Statistical Machine Learning provides algorithms that let the data define the rules.

**Key Takeaways:**
1. **KNN:** A simple, distance-based algorithm. It **requires** scaling (standardization) to work correctly.
2. **Decision Trees:** Excellent for finding non-linear splits and highly interpretable, but easily overfit the training data.
3. **Random Forests:** Fixes tree overfitting by using Bootstrap samples (Bagging) and random feature subsets to create a highly accurate, generalized "wisdom of the crowd".
4. **Boosting:** The reigning champion of tabular data. By focusing sequentially on previous errors, it builds an incredibly precise model, though it requires careful tuning of its learning rate.